# 01 - Exploratory Data Analysis
## InclusionScore AI - Alternate Credit Scoring

**Dataset:** Give Me Some Credit (`cs-training.csv`)  
**Goal:** Understand data shape, distributions, missing values, correlations, and plan cleaning.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json

from src.data_loader import load_primary_dataset
from src.utils import summarize_df

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Load Dataset

In [ ]:
df = load_primary_dataset()
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# JSON-friendly summary
summary = summarize_df(df)
print(json.dumps({k: v for k, v in summary.items() if k != 'describe'}, indent=2))

## 2. Target Distribution

The target `SeriousDlqin2yrs` is a binary flag (1 = experienced 90+ days past due).

In [ ]:
counts = df['SeriousDlqin2yrs'].value_counts()
print(f'No Default (0): {counts[0]:,} ({counts[0]/len(df)*100:.1f}%)')
print(f'Default    (1): {counts[1]:,} ({counts[1]/len(df)*100:.1f}%)')
print(f'Imbalance ratio: 1:{counts[0]//counts[1]}')

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['No Default (0)', 'Default (1)'], counts.values, color=['#2196F3', '#f44336'])
ax.set_ylabel('Count')
ax.set_title('Target Distribution: SeriousDlqin2yrs')
for i, v in enumerate(counts.values):
    ax.text(i, v + 1000, f'{v:,} ({v/len(df)*100:.1f}%)', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

## 3. Missing Values

In [ ]:
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(2)
missing_df = pd.DataFrame({'count': null_counts, 'pct': null_pct}).sort_values('count', ascending=False)
print(missing_df[missing_df['count'] > 0])

fig, ax = plt.subplots(figsize=(8, 4))
cols_with_nulls = null_pct[null_pct > 0].sort_values(ascending=False)
ax.barh(cols_with_nulls.index, cols_with_nulls.values, color='#FF9800')
ax.set_xlabel('Missing (%)')
ax.set_title('Missing Values by Column')
for i, v in enumerate(cols_with_nulls.values):
    ax.text(v + 0.3, i, f'{v:.1f}%', va='center', fontsize=9)
plt.tight_layout()
plt.show()

## 4. Correlation Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Matrix')
plt.tight_layout()
plt.show()

print('\nCorrelation with target:')
print(corr['SeriousDlqin2yrs'].drop('SeriousDlqin2yrs').sort_values(ascending=False))

## 5. Feature Distributions by Default Status

In [ ]:
# Age distribution
fig, ax = plt.subplots(figsize=(8, 4))
df[df['SeriousDlqin2yrs']==0]['age'].hist(bins=50, alpha=0.6, label='No Default', color='#2196F3', ax=ax, density=True)
df[df['SeriousDlqin2yrs']==1]['age'].hist(bins=50, alpha=0.6, label='Default', color='#f44336', ax=ax, density=True)
ax.set_xlabel('Age')
ax.set_ylabel('Density')
ax.set_title('Age Distribution by Default Status')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots for key features (clipped at 99th percentile for readability)
features_to_plot = [
    'RevolvingUtilizationOfUnsecuredLines',
    'DebtRatio',
    'MonthlyIncome',
    'NumberOfTime30-59DaysPastDueNotWorse',
    'NumberOfTimes90DaysLate',
    'NumberOfTime60-89DaysPastDueNotWorse',
]
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for idx, feat in enumerate(features_to_plot):
    ax = axes[idx // 3][idx % 3]
    p99 = df[feat].quantile(0.99)
    clipped = df[feat].clip(upper=p99)
    sns.boxplot(x=df['SeriousDlqin2yrs'], y=clipped, hue=df['SeriousDlqin2yrs'],
                palette=['#2196F3', '#f44336'], ax=ax, legend=False)
    ax.set_title(feat[:30], fontsize=9)
    ax.set_xlabel('Default')
plt.suptitle('Feature Distributions by Default (clipped at 99th pctile)', fontsize=12)
plt.tight_layout()
plt.show()

## 6. Delinquency Features Analysis

The three past-due columns have suspicious sentinel values of 96 and 98.

In [ ]:
delinq_cols = [
    'NumberOfTime30-59DaysPastDueNotWorse',
    'NumberOfTime60-89DaysPastDueNotWorse',
    'NumberOfTimes90DaysLate'
]

for col in delinq_cols:
    sentinel = df[col].isin([96, 98]).sum()
    print(f'{col}: {sentinel} rows with value 96 or 98 (likely sentinel/error)')

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for idx, col in enumerate(delinq_cols):
    ax = axes[idx]
    capped = df[col].clip(upper=10)
    capped.value_counts().sort_index().plot(kind='bar', ax=ax, color='#607D8B')
    ax.set_title(col[:35], fontsize=9)
    ax.set_xlabel('Count')
plt.suptitle('Delinquency Feature Distributions (capped at 10)', fontsize=12)
plt.tight_layout()
plt.show()

## 7. Outlier Analysis

In [ ]:
print('=== Outlier/Anomaly Summary ===')
print(f'age == 0: {(df.age == 0).sum()} rows (invalid age, will drop)')
print(f'RevolvingUtilization > 1: {(df.RevolvingUtilizationOfUnsecuredLines > 1).sum()} rows')
print(f'  > 1 means utilization exceeds credit limit; will cap at a reasonable threshold')
print(f'DebtRatio > 10: {(df.DebtRatio > 10).sum()} rows')
print(f'  Extreme ratios likely from missing/low income; will cap at 99th percentile')
print(f'Delinquency == 96 or 98: {(df[delinq_cols] >= 96).any(axis=1).sum()} rows')
print(f'  Sentinel values; will cap at a reasonable max (e.g., 15)')

## 8. Cleaning & Feature Engineering Plan

Based on this EDA, the cleaning plan is:

| Step | Action |
|------|--------|
| Drop rows | `age == 0` (1 row) |
| Impute `MonthlyIncome` | Median imputation (19.8% missing) |
| Impute `NumberOfDependents` | Mode imputation (2.6% missing, mode = 0) |
| Cap `RevolvingUtilizationOfUnsecuredLines` | Clip at 1.0 (values > 1 are anomalous) |
| Cap `DebtRatio` | Clip at 99th percentile |
| Cap delinquency columns | Replace 96/98 sentinel values with max observed (e.g., cap at 15) |
| No protected attributes present | Dataset has no gender/ethnicity/religion columns |

**Engineered features (Phase 2):**
- `TotalDelinquencies`: sum of all past-due columns
- `HasDelinquency`: binary flag (any past-due > 0)
- `IncomeToDebtRatio`: `MonthlyIncome / (DebtRatio + 1)`
- `AgeBin`: age buckets for non-linear effects
- `CreditUtilizationBucket`: binned revolving utilization